# Python Internals and Performance: A Coherent Mental Model

Maps to `design3.md` Phase 1.

This notebook is about how Python behaves underneath everyday syntax. The point is not to memorize implementation trivia. The point is to build a mental model that explains the bugs people actually hit in real code: aliasing, mutable defaults, late binding, bad container choices, confusing imports, bad timing measurements, and wrong concurrency choices.


## How To Use This Notebook

This notebook has a strict sequence. Do not skip around.

1. Names, objects, identity, equality
2. Mutation, aliasing, shallow vs deep copy
3. Function defaults and when they are evaluated
4. Scope, closures, and late binding
5. Hashing, hashability, and why `dict` / `set` work
6. List behavior and why `deque` exists
7. Iterators and generators
8. Imports and module caching
9. CPython memory model: reference counting and cyclic GC
10. Timing vs profiling
11. GIL, threads, and processes

If the sequence is wrong, the notebook stops making sense. That was one of the real problems in the earlier version.


## 1. Names, Objects, Identity, and Equality

Python variables are **names bound to objects**.

This is the first mental model to get right.

Important distinctions:
- **name**: a label like `x` or `trade`
- **object**: the thing in memory the name refers to
- **type**: what kind of object it is (`list`, `dict`, `str`, etc.)
- **identity**: which exact object it is (`id(obj)`)
- **equality**: whether two objects compare as having the same value (`==`)

Two names can point at the same object. Two different objects can compare equal.

Interview question:
- What is the difference between `is` and `==`?
  `is` checks identity. `==` checks value equality according to the type's comparison rules.


In [ ]:
record_a = {"symbol": "AAPL", "prices": [100.0, 101.5]}
record_b = record_a
record_c = {"symbol": "AAPL", "prices": [100.0, 101.5]}

identity_demo = {
    "record_a_is_record_b": record_a is record_b,
    "record_a_equals_record_b": record_a == record_b,
    "record_a_is_record_c": record_a is record_c,
    "record_a_equals_record_c": record_a == record_c,
    "id_record_a": id(record_a),
    "id_record_b": id(record_b),
    "id_record_c": id(record_c),
}

identity_demo


## 2. Mutation, Aliasing, and Copying

Aliasing means multiple names refer to the same object. This matters most when the object is mutable.

Mutation changes an object in place.
Rebinding changes which object a name points to.

This explains a large fraction of Python bugs in data pipelines and app code.

Copying rules:
- assignment does **not** copy
- `copy.copy(x)` makes a **shallow** copy
- `copy.deepcopy(x)` makes a **deep** copy

Shallow copy means the outer container is new, but nested objects are still shared.
Deep copy means nested objects are recursively copied too.


In [ ]:
import copy

base_config = {
    "source": "vendor_a",
    "symbols": ["AAPL", "MSFT"],
    "thresholds": {"warn": 0.01, "error": 0.05},
}

assigned = base_config
shallow = copy.copy(base_config)
deep = copy.deepcopy(base_config)

assigned["symbols"].append("NVDA")
shallow["thresholds"]["warn"] = 0.02
deep["symbols"].append("AMZN")

copy_demo = {
    "base_config": base_config,
    "assigned": assigned,
    "shallow": shallow,
    "deep": deep,
    "base_symbols_is_shallow_symbols": base_config["symbols"] is shallow["symbols"],
    "base_thresholds_is_shallow_thresholds": base_config["thresholds"] is shallow["thresholds"],
    "base_symbols_is_deep_symbols": base_config["symbols"] is deep["symbols"],
}

copy_demo


### Interview Questions

- Why does assignment not copy a list or dict?
- What is a shallow copy of a nested dictionary?
- When is shallow copy enough, and when is deep copy safer?
- What is the practical danger of aliasing in pipeline configuration objects?


## 3. Function Defaults: When Are They Evaluated?

Python evaluates default argument expressions **once**, when the function is defined, not every time the function is called.

This is the real reason mutable default arguments are dangerous.

Bad pattern:
- `def f(x, bucket=[]): ...`

Why it is bad:
- the same list object is reused across calls

Correct pattern:
- use `None` as the sentinel and create a new object inside the function


In [ ]:
def append_bad(value, bucket=[]):
    bucket.append(value)
    return bucket


def append_good(value, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(value)
    return bucket

mutable_default_demo = {
    "bad_first": append_bad("AAPL"),
    "bad_second": append_bad("MSFT"),
    "good_first": append_good("AAPL"),
    "good_second": append_good("MSFT"),
}

mutable_default_demo


### Interview Questions

- Why does `append_bad` keep state between calls?
- Why is `None` the standard sentinel for mutable default arguments?
- Is the bug caused by lists specifically, or by when defaults are evaluated?


## 4. Scope, Closures, and Late Binding

The usual summary is LEGB:
- Local
- Enclosing
- Global
- Built-in

That tells you where Python looks for a name.

The closure bug you must know:
- functions created in a loop often capture the **name**, not the value at that moment
- when the function is called later, the loop variable may have its final value

This is called **late binding**.


In [ ]:
def make_multipliers_bad():
    return [lambda x: i * x for i in range(3)]


def make_multipliers_good():
    return [lambda x, i=i: i * x for i in range(3)]

closure_demo = {
    "bad_results": [fn(10) for fn in make_multipliers_bad()],
    "good_results": [fn(10) for fn in make_multipliers_good()],
}

closure_demo


### Interview Questions

- What does "late binding" mean in Python closures?
- Why does `i=i` in the lambda fix the problem?
- What is the difference between a local variable and an enclosing-scope variable?


## 5. Hashing, Hashability, and Why `dict` / `set` Work

At a practical level, dictionaries and sets are hash-table-based structures.

That gives you these day-to-day rules:
- repeated key lookup in a dictionary is usually much better than scanning a list
- repeated membership testing in a set is usually much better than scanning a list
- keys and set elements must be **hashable**

Hashable usually means:
- the object has a stable hash value during its lifetime
- it can be compared for equality consistently

Strings, numbers, and tuples of hashable values are hashable.
Lists and dictionaries are not.


In [ ]:
import timeit

values = list(range(100_000))
value_set = set(values)
target = 99_999

hash_demo = {
    "hash_string": hash("AAPL"),
    "hash_tuple": hash(("AAPL", "NASDAQ")),
    "list_membership_seconds": round(timeit.timeit("target in values", globals=globals(), number=2000), 6),
    "set_membership_seconds": round(timeit.timeit("target in value_set", globals=globals(), number=2000), 6),
}

try:
    hash(["AAPL", "NASDAQ"])
except TypeError as exc:
    hash_demo["list_hash_error"] = str(exc)

hash_demo


### Interview Questions

- What does it mean for an object to be hashable?
- Why can a tuple often be a dict key while a list cannot?
- Why are `dict` and `set` usually better than lists for repeated lookup or membership testing?
- Why is the answer "usually" rather than "always"?


## 6. Lists as Dynamic Arrays, and Why `deque` Exists

You do not need CPython source code to reason about lists correctly.

Useful mental model:
- a Python list behaves like a dynamic array
- appending at the end is efficient in normal use
- removing from the front forces the remaining elements to shift left

That leads to the rule:
- `list.append()` and `list.pop()` are good stack operations
- `list.pop(0)` is a poor queue operation
- `collections.deque` exists for queue-like workloads with fast operations at both ends


In [ ]:
from collections import deque

queue_list = list(range(10_000))
queue_deque = deque(range(10_000))

list_front_pop = timeit.timeit("q = queue_list.copy(); q.pop(0)", globals=globals(), number=1000)
deque_left_pop = timeit.timeit("q = queue_deque.copy(); q.popleft()", globals=globals(), number=1000)

list_behavior_demo = {
    "list_front_pop_seconds": round(list_front_pop, 6),
    "deque_left_pop_seconds": round(deque_left_pop, 6),
}

list_behavior_demo


### Interview Questions

- Why is a list good as a stack but poor as a queue?
- What workload is `deque` designed for?
- What does it mean when someone says `list.append` is "amortized O(1)"?


## 7. Iterators and Generators

A generator does not build all values up front. It yields them lazily as needed.

That gives you two important ideas:
- generators can reduce peak memory usage
- generators are single-pass iterators; once consumed, they are exhausted

This is one of the most useful Python ideas in data work because many workloads are naturally stream-like.


In [ ]:
import sys

materialized = [x * 2 for x in range(10_000)]
lazy = (x * 2 for x in range(10_000))

first_three = [next(lazy) for _ in range(3)]
remaining_sum = sum(lazy)
exhausted_again = list(lazy)

generator_demo = {
    "list_size_bytes": sys.getsizeof(materialized),
    "generator_object_size_bytes": sys.getsizeof((x * 2 for x in range(10_000))),
    "first_three": first_three,
    "remaining_sum": remaining_sum,
    "exhausted_again": exhausted_again,
}

generator_demo


### Interview Questions

- What is the difference between an iterable and an iterator?
- Why can generators reduce memory usage?
- Why is a generator not automatically the right answer for every workload?
- What does it mean that generators are single-pass?


## 8. Imports and Module Caching

Imports matter because real Python systems are built from modules and packages.

Key facts:
- importing a module executes its top-level code the first time
- loaded modules are cached in `sys.modules`
- later imports usually reuse the cached module object

Practical rule:
- avoid expensive side effects at import time
- avoid network calls, huge file loads, and surprising stateful behavior at module top level


In [ ]:
import importlib
import math
import sys

math_again = importlib.import_module("math")

import_demo = {
    "math_in_sys_modules": "math" in sys.modules,
    "same_module_object": math is math_again,
    "module_name": math.__name__,
}

import_demo


### Interview Questions

- Why is repeated import usually cheap?
- What kinds of side effects are dangerous at import time?
- What role does `sys.modules` play in the import system?


## 9. CPython Memory Model: Reference Counting and Cyclic GC

This section is explicitly about **CPython**, which is the implementation most people use.

Useful mental model:
- CPython mainly manages object lifetime with reference counting
- cycles cannot be cleaned up by reference counting alone
- the cyclic garbage collector exists to detect and collect unreachable cycles

This is why both statements can be true:
- many objects disappear immediately when their last reference goes away
- some cyclic objects need the GC to clean them up later


In [ ]:
import gc
import sys

payload = {"symbol": "AAPL", "venue": "NASDAQ"}
ref_before = sys.getrefcount(payload)
alias = payload
ref_after_alias = sys.getrefcount(payload)
del alias
ref_after_delete = sys.getrefcount(payload)

gc_demo = {
    "gc_enabled": gc.isenabled(),
    "gc_thresholds": gc.get_threshold(),
    "ref_before_alias": ref_before,
    "ref_after_alias": ref_after_alias,
    "ref_after_delete": ref_after_delete,
}

gc_demo


### Interview Questions

- What is reference counting?
- Why is `sys.getrefcount` only useful for relative understanding, not exact application logic?
- Why do cycles need a garbage collector in addition to reference counting?


## 10. Timing vs Profiling

These answer different questions.

Use `timeit` when the question is:
- which of these small code snippets is faster on this machine?

Use `cProfile` when the question is:
- where is the program spending time overall?

Common mistake:
- people use a single `time.time()` print for everything and then optimize the wrong thing


In [ ]:
generator_expr_time = timeit.timeit(
    "sum(x * x for x in range(10_000))",
    globals=globals(),
    number=500,
)

list_comp_time = timeit.timeit(
    "sum([x * x for x in range(10_000)])",
    globals=globals(),
    number=500,
)

{
    "generator_expr_seconds": round(generator_expr_time, 6),
    "list_comp_seconds": round(list_comp_time, 6),
}


In [ ]:
import cProfile
import pstats


def sum_squares_list(n: int) -> int:
    values = [x * x for x in range(n)]
    return sum(values)


profiler = cProfile.Profile()
profiler.enable()
sum_squares_list(50_000)
profiler.disable()

stats = pstats.Stats(profiler).sort_stats("cumulative")
stats.print_stats(8)


### Interview Questions

- When would you use `timeit` instead of `cProfile`?
- Why can microbenchmarks mislead you?
- Why is algorithm choice often more important than tiny speed differences in a single expression?


## 11. GIL, Threads, and Processes

Now that the earlier pieces are in place, this section makes more sense.

In standard CPython:
- only one thread executes Python bytecode at a time in a process
- this is the GIL story in one sentence

But that does **not** mean threads are useless.

Practical decision rule:
- I/O-bound work: threads can help because waiting on I/O can release execution to other threads
- CPU-bound pure Python work: processes are often the better choice for parallelism across cores

This is a decision rule, not a slogan.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time


def fetch_like(delay: float) -> float:
    time.sleep(delay)
    return delay


delays = [0.2, 0.2, 0.2, 0.2]

start = time.perf_counter()
sequential = [fetch_like(delay) for delay in delays]
sequential_elapsed = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    threaded = list(pool.map(fetch_like, delays))
threaded_elapsed = time.perf_counter() - start

{
    "sequential_elapsed": round(sequential_elapsed, 3),
    "threaded_elapsed": round(threaded_elapsed, 3),
    "results_match": sequential == threaded,
}


### Interview Questions

- What does the GIL mean in practical terms?
- Why can threads help for I/O-bound work but not necessarily for CPU-bound pure Python loops?
- When would you choose processes instead of threads?
- What is the difference between concurrency and parallelism?


## Exit Checklist

Do not move on until you can answer these without searching:
- What is the difference between `is` and `==`?
- What is aliasing, and why does it matter for mutable objects?
- What is the difference between assignment, shallow copy, and deep copy?
- Why are mutable default arguments dangerous?
- What is late binding in a closure?
- What does it mean for an object to be hashable?
- Why are `dict` and `set` usually better than lists for repeated lookup or membership?
- Why is `deque` better than `list` for queue workloads?
- What is the difference between an iterator and a generator?
- What does `sys.modules` do?
- What part of this notebook is CPython-specific?
- When should you use `timeit` vs `cProfile`?
- What does the GIL mean for threads and processes?


## Official References Used To Build This Notebook

- Python Language Reference: Data Model
  https://docs.python.org/3/reference/datamodel.html
- Python Standard Library: Built-in Types
  https://docs.python.org/3/library/stdtypes.html
- Python FAQ: Programming
  https://docs.python.org/3/faq/programming.html
- Python Standard Library: `copy`
  https://docs.python.org/3/library/copy.html
- Python Standard Library: `gc`
  https://docs.python.org/3/library/gc.html
- Python Standard Library: `timeit`
  https://docs.python.org/3/library/timeit.html
- Python Standard Library: `profile` and `cProfile`
  https://docs.python.org/3/library/profile.html
- Python Standard Library: `threading`
  https://docs.python.org/3/library/threading.html
- Python Standard Library: `multiprocessing`
  https://docs.python.org/3/library/multiprocessing.html
